# `run_eval_pipeline.ipynb` — speculative-decoding eval driver (RunAI)

Drives the eval contract documented in `CLAUDE.md` (Eval contract — `scripts/evaluate_sd.py`):
loads the HF target and (when `draft` is set) the HF draft, runs the
instrumented loop in `src/kdsd/sd/instrument.py`, and writes
`/scratch/cs552-results/<run_name>/{eval_summary.json, generations.jsonl, timing.json,
config.yaml}` (per `rcp_support/README.md` §"Storage layout"). The instrumented
loop is the sole source of `acceptance_rate`, `avg_accepted_tokens`, per-prompt
`accepted_lens`, and the wall-clock speedup.

## How to run

1. From your laptop, submit the interactive Jupyter pod with `notebooks/submit.sh`
   (set `GASPAR` and `GROUP` first, see `rcp_support/README.md`).
2. `runai port-forward <job-name> --port 8888:8888` — open
   `http://localhost:8888` (token `cs552`).
3. Inside Jupyter, clone this repo under `/scratch/` and open this notebook from
   `/scratch/<repo>/notebooks/run_eval_pipeline.ipynb`.
4. Edit the **Configuration** cell, then `Run All`.

## 1. Environment

`HF_HOME=/scratch/hf_cache` is already set by `notebooks/submit.sh`. We re-assert
it here so the notebook also runs from a `runai bash` shell or a non-default
launcher. `evaluate_sd.py` reads `cfg.hf_cache` and re-exports `HF_HOME` /
`HF_HUB_CACHE` / `HF_DATASETS_CACHE` before importing transformers.

In [14]:
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)
print("repo root:", REPO)

os.environ.setdefault("HF_HOME", "/scratch/hf_cache")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

repo root: /scratch/cs552-mnlp-youyang
HF_HOME: /scratch/hf_cache


In [15]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 40439 MiB


## 2. Bootstrap the `kdsd` package

The course image already ships torch 2.8, transformers 4.57, and hydra-core. We
only need the local `kdsd` package on the import path.

In [16]:
# skip the install because current env already is set up.
# !pip install --quiet -e .

## 3. Configuration

Edit these for your run. `hydra_overrides` is forwarded verbatim to
`evaluate_sd.py` (same grammar as `uv run python scripts/evaluate_sd.py
<overrides>`).

In [ ]:
RUN_NAME = "spec_smoke"
DRAFT = "Qwen/Qwen2.5-0.5B-Instruct"
# PROMPTS_JSONL = "data/processed/eval.jsonl"
PROMPTS_LIMIT = 20
GAMMA = 4
MAX_NEW_TOKENS = 256
N_REPEATS = 1
N_WARMUP = 1

hydra_overrides = [
    f"run_name={RUN_NAME}",
    f"draft={DRAFT}",
    # f"prompts.jsonl={PROMPTS_JSONL}",
    f"prompts.limit={PROMPTS_LIMIT}",
    f"runtime.gamma={GAMMA}",
    f"runtime.max_new_tokens={MAX_NEW_TOKENS}",
    f"eval.n_repeats={N_REPEATS}",
    f"eval.n_warmup={N_WARMUP}",
]

# Mirrors configs/config.yaml `results_dir: /scratch/cs552-results/${run_name}`.
RESULTS_DIR = Path("/scratch/cs552-results") / RUN_NAME
print("hydra overrides:", hydra_overrides)
print("results dir:", RESULTS_DIR)

## 4. Run the eval

Streams `evaluate_sd.py` stdout into the cell.

In [18]:
import subprocess

EVAL_SCRIPT = REPO / "scripts" / "evaluate_sd.py"

cmd = [sys.executable, str(EVAL_SCRIPT), *hydra_overrides]
print(f"\n>>> {' '.join(cmd)}\n", flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
print(f"\n<<< exit code: {rc}", flush=True)
if rc != 0:
    raise RuntimeError(f"evaluate_sd.py failed (rc={rc})")


>>> /usr/bin/python3 /scratch/cs552-mnlp-youyang/scripts/evaluate_sd.py run_name=spec_smoke draft=Qwen/Qwen2.5-0.5B-Instruct prompts.limit=20 runtime.gamma=4 runtime.max_new_tokens=256

2026-05-08 09:35:45,162 INFO    kdsd.evaluate_sd :: resolved config:
model:
  target: Qwen/Qwen2.5-3B-Instruct
  draft_default: Qwen/Qwen2.5-0.5B-Instruct
  dtype: bfloat16
  device: cuda
  attn_impl: sdpa
  trust_remote_code: false
eval:
  n_warmup: 1
  n_repeats: 1
  run_vanilla_baseline: true
  write_generations: true
runtime:
  mode: sampling
  temperature: 1.0
  top_p: 0.9
  gamma: 4
  max_new_tokens: 256
benchmark:
  benchmarks: []
run_name: spec_smoke
seed: 42
draft: Qwen/Qwen2.5-0.5B-Instruct
prompts:
  jsonl: null
  hf_dataset:
    name: HuggingFaceH4/ultrachat_200k
    split: test_sft
    prompt_field: prompt
  limit: 20
results_dir: results/${run_name}
hf_cache: ${oc.env:HF_HOME,'~/.cache/huggingface'}

2026-05-08 09:35:45,162 INFO    kdsd.evaluate_sd :: HF_HOME=/scratch/hf_cache

Loading ch

## 5. Inspect the summary

The schema is frozen in `CLAUDE.md` (Eval contract). `engines.hf` preserves the
raw HF-side numbers alongside the top-level fields.

In [19]:
import json

summary_path = RESULTS_DIR / "eval_summary.json"
summary = json.loads(summary_path.read_text())

top_keys = (
    "acceptance_rate", "avg_accepted_tokens",
    "vanilla_time_s", "sd_time_s", "speedup", "tokens_per_second",
    "n_prompts", "n_warmup", "n_repeats",
)
for k in top_keys:
    print(f"{k:25s} {summary.get(k)}")

print("\nengines:")
print(json.dumps(summary.get("engines", {}), indent=2))

acceptance_rate           0.47937245787332944
avg_accepted_tokens       1.9174898314933178
vanilla_time_s            140.78342496976256
sd_time_s                 266.85367163596675
speedup                   0.5301144635261005
tokens_per_second         18.72187093912421
n_prompts                 20
n_warmup                  1
n_repeats                 1

engines:
{
  "hf": {
    "sd_time_s": 266.85367163596675,
    "vanilla_time_s": 140.78342496976256,
    "tokens_per_second": 18.72187093912421,
    "speedup": 0.5301144635261005,
    "acceptance_rate": 0.47937245787332944,
    "avg_accepted_tokens": 1.9174898314933178,
    "n_outer_steps": 1721,
    "target_calls": 3442,
    "draft_calls": 8605,
    "draft_forward_s": 152.55314103221892,
    "target_forward_s": 104.2545477142334,
    "batched": false
  }
}


In [20]:
gens_path = RESULTS_DIR / "generations.jsonl"
with gens_path.open() as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        rec = json.loads(line)
        prompt = rec.get("prompt", "")
        gen = rec.get("generation", "")
        print(f"--- record {i} ---")
        print("prompt   :", prompt[:200].replace("\n", " "))
        print("generation:", gen[:200].replace("\n", " "))
        print("accepted_lens:", rec.get("accepted_lens"))
        print()

--- record 0 ---
prompt   : How does the author propose to fix the problem of science alienation in our educational system? What changes does she suggest should be made to science education? Answer according to: Science educatio
generation: The author proposes a solution to the problem of science alienation in our educational system by suggesting a significant overhaul of K-12 science education. Here are the key changes she advocates for
accepted_lens: [0, 2, 0, 4, 4, 4, 2, 4, 4, 2, 4, 1, 0, 1, 1, 4, 4, 0, 2, 0, 0, 4, 2, 4, 2, 0, 0, 0, 1, 4, 1, 1, 4, 0, 2, 4, 4, 4, 1, 4, 4, 0, 1, 0, 1, 0, 4, 2, 2, 4, 0, 4, 3, 4, 3, 1, 3, 0, 4, 1, 3, 1, 1, 0, 0, 0, 3, 4, 1, 3, 2, 4, 0, 0, 0, 1, 2, 3, 3, 4, 4, 3, 4, 0]

--- record 1 ---
prompt   : Rice tolerance to suboptimal low temperatures relies on the maintenance of the photosynthetic capacity. Gazquez, A., Vilas, J. M., Colman Lerner, J. E., Maiale, S. J., Calzadilla, P. I., Menendez, A. 
generation: The study focused on two contrasting rice cultiv